# SOXL 稳定超额前沿研究

本卡在 qlib 原生链路上评估一族**预先登记、无收益预测、因果使用**的 SOXX/SOXL/现金暴露规则，
检验它们能否在收益、回撤、Sharpe 三方面同时稳定超过 SOXX buy-and-hold。
选择只使用 development 区间；reused diagnostic 与新的 forward 区间仅用于评估。


## 1. 环境与预登记计划

先锁定 plan.json 的 sha256，再执行任何回放。


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
CARD = ROOT / 'research' / 'soxl_stable_excess_frontier_20261010'
plan_bytes = (CARD / 'plan.json').read_bytes()
plan = json.loads(plan_bytes)
plan_sha256 = hashlib.sha256(plan_bytes).hexdigest()
print('plan sha256:', plan_sha256)
print('provider:', plan['provider_uri'], '| as_of:', plan['as_of'])
print('segments:', plan['segments'])
print('candidates:', len(plan['candidate_family']))


plan sha256: 9939a0a6a0140936c03cddc3ef363c57192d962f5f1968714439557178e8c685
provider: data/qlib/soxl | as_of: 2026-10-08
segments: {'development': ['2016-01-04', '2022-12-30'], 'reused_diagnostic': ['2023-01-03', '2026-09-25'], 'forward': ['2026-09-26', '2026-10-08']}
candidates: 9


## 2. 运行预登记候选族（qlib Recorder）

工作流通过 `qlib.init` + `qlib.data.D.features` 读取 SOXL/SOXX，日内/状态均为过去可得，
预测与结果写入 qlib Recorder，卡根只保留小型 sidecar。


In [2]:
from src.workflow import stable_excess_frontier as workflow
from src.workflow.stable_excess_frontier import write_card_files

summary = workflow.run_stable_excess_frontier(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'])
print('selected policy:', summary['selected_policy'])
print('promotion_eligible:', summary['promotion_eligible'])
print('recorder_id:', summary['recorder_id'])
print('provider_uri:', summary['provider_uri'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[10795:MainThread](2026-10-09 16:06:50,238) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


2026/10/09 16:06:50 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[10795:MainThread](2026-10-09 16:06:50,615) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[10795:MainThread](2026-10-09 16:06:50,615) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


...

[10795:MainThread](2026-10-09 16:06:53,515) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name soxl_stable_excess_frontier_20261010.


[10795:MainThread](2026-10-09 16:06:53,517) INFO - qlib.workflow - [exp.py:258] - Experiment 550974314769632755 starts running ...


[10795:MainThread](2026-10-09 16:06:53,570) INFO - qlib.workflow - [recorder.py:345] - Recorder f0d3bdc1e5b541d2a188c6a4486417aa starts running under Experiment 550974314769632755 ...


[10795:MainThread](2026-10-09 16:06:53,645) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


decision: no_stable_excess_in_registered_family
selected policy: trend_vol_switch_v35
promotion_eligible: False
recorder_id: f0d3bdc1e5b541d2a188c6a4486417aa
provider_uri: /Users/rziiiii/dev/workspace/data/qlib/soxl


## 3. 候选对比（10bp）


In [3]:
import pandas as pd
comp = pd.read_csv(CARD / 'policy_comparison.csv')
cols = ['window','policy','return_pct','max_drawdown_pct','benchmark_return_pct',
        'benchmark_max_drawdown_pct','sharpe_difference','stable_excess',
        'positive_year_count','year_count','average_soxl_weight','average_cash_weight']
display(comp[cols].round(3))


,window,policy,return_pct,max_drawdown_pct,benchmark_return_pct,benchmark_max_drawdown_pct,sharpe_difference,stable_excess,positive_year_count,year_count,average_soxl_weight,average_cash_weight
0,development,soxx_buy_hold,291.078,-46.245,291.078,-46.245,0.000,False,0,7,0.000,0.000
1,development,vol_target_soxx_v15,114.686,-20.337,291.078,-46.245,-0.014,False,1,7,0.000,0.380
2,development,vol_target_soxx_v20,170.088,-25.895,291.078,-46.245,0.034,False,1,7,0.000,0.237
3,development,vol_target_soxx_v25,213.744,-31.656,291.078,-46.245,0.050,False,1,7,0.000,0.149
4,development,vol_target_soxx_v30,243.870,-37.027,291.078,-46.245,0.048,False,1,7,0.000,0.092
5,development,trend_vol_switch_v20,398.572,-47.210,291.078,-46.245,0.021,False,4,7,0.313,0.221
6,development,trend_vol_switch_v25,258.426,-62.774,291.078,-46.245,-0.148,False,3,7,0.496,0.221
7,development,trend_vol_switch_v30,690.099,-47.550,291.078,-46.245,0.053,False,5,7,0.583,0.221
8,development,trend_vol_switch_v35,1198.681,-52.518,291.078,-46.245,0.145,False,5,7,0.680,0.221
9,reused_diagnostic,soxx_buy_hold,397.603,-41.670,397.603,-41.670,0.000,False,0,4,0.000,0.000


## 4. 选中策略的年度分解


In [4]:
ann = pd.read_csv(CARD / 'annual_metrics.csv')
display(ann.round(4))
for window in ['development','reused_diagnostic','forward']:
    row = summary['window_comparison']
print('forward verdict:', json.dumps(summary['selected_forward_verdict'], indent=2))


,window,policy,year,sessions,strategy_return_pct,benchmark_return_pct,wealth_ratio_vs_soxx,average_soxl_weight,average_cash_weight
0,development,trend_vol_switch_v35,2016,252,103.6615,37.9192,1.4767,0.8016,0.1984
1,development,trend_vol_switch_v35,2017,251,161.0071,42.1284,1.8364,1.0000,0.0000
2,development,trend_vol_switch_v35,2018,251,-36.1841,-9.4599,0.7048,0.6215,0.2669
3,development,trend_vol_switch_v35,2019,252,119.2595,62.2910,1.3510,0.8611,0.1151
4,development,trend_vol_switch_v35,2020,253,64.7301,47.3292,1.1181,0.6443,0.1107
5,development,trend_vol_switch_v35,2021,252,95.3758,46.4816,1.3338,0.7778,0.0000
6,development,trend_vol_switch_v35,2022,250,-45.7487,-37.0855,0.8623,0.0480,0.8640
7,reused_diagnostic,trend_vol_switch_v35,2023,250,87.0758,60.8483,1.1631,0.7720,0.0560
8,reused_diagnostic,trend_vol_switch_v35,2024,252,18.2243,17.1261,1.0094,0.5754,0.1944
9,reused_diagnostic,trend_vol_switch_v35,2025,250,10.5806,44.6776,0.7643,0.4640,0.3680


forward verdict: {
  "beats_return_and_drawdown": false,
  "beats_sharpe": false,
  "positive_year_count": 0,
  "year_count": 1,
  "minimum_positive_years": 1,
  "annual_win_rate": 0.0,
  "stable_excess": false,
  "reference": "SOXX buy-and-hold, identical sessions and cost model",
  "scope": "reused_history_and_forward_diagnostic_not_production_evidence"
}


## 5. 中文结论


In [5]:
outer = summary['selected_outer_10bps']['strategy']
fwd = summary['selected_forward']['strategy']
print(f"选中策略 {summary['selected_policy']}；候选族 {summary['candidate_count']} 条，无收益预测。")
print(f"development 选择仅使用 {summary['segments']['development']}。")
print(f"复用区间 {summary['segments']['reused_diagnostic']}：收益 {outer['return_pct']:.2f}%，"
      f"回撤 {outer['max_drawdown_pct']:.2f}%，Sharpe {outer['sharpe']:.2f}。")
print(f"前向区间 {summary['segments']['forward']}：收益 {fwd['return_pct']:.2f}%，"
      f"回撤 {fwd['max_drawdown_pct']:.2f}%，Sharpe {fwd['sharpe']:.2f}。")
print('前向同时超过 SOXX 收益与回撤:', summary['selected_forward_verdict']['beats_return_and_drawdown'])
print('是否晋升:', summary['promotion_eligible'], '| 决定:', summary['decision'])
print('qlib Recorder:', summary['recorder_id'])


选中策略 trend_vol_switch_v35；候选族 9 条，无收益预测。
development 选择仅使用 ['2016-01-04', '2022-12-30']。
复用区间 ['2023-01-03', '2026-09-25']：收益 348.00%，回撤 -68.48%，Sharpe 0.93。
前向区间 ['2026-09-26', '2026-10-08']：收益 0.25%，回撤 -4.45%，Sharpe 0.41。
前向同时超过 SOXX 收益与回撤: False
是否晋升: False | 决定: no_stable_excess_in_registered_family
qlib Recorder: f0d3bdc1e5b541d2a188c6a4486417aa
